# LeetCode 2207 — Maximize Number of Subsequences in a String bằng Python

Link đề: https://leetcode.com/problems/maximize-number-of-subsequences-in-a-string/ · Độ khó: **Medium** · Chủ đề: String, Greedy, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def maximumSubsequenceCount(self, text: str, pattern: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Đếm số lần một chuỗi 2 ký tự `ab` xuất hiện như **dãy con** trong `O(n)` bằng kỹ thuật "mỗi `b` ghép với mọi `a` đứng trước nó".
2. Phân tích **ảnh hưởng của việc chèn một ký tự** tại vị trí `p`: chỉ phụ thuộc số `a` bên trái hoặc số `b` bên phải vị trí đó.
3. Rút ra **tham lam**: chèn `pattern[0]` ở **đầu** hoặc `pattern[1]` ở **cuối** luôn tối ưu.
4. Xử lý trường hợp đặc biệt `pattern[0] == pattern[1]` (thứ tự cập nhật bộ đếm rất quan trọng).
5. Kiểm thử đối chiếu brute force thử mọi vị trí chèn trên dữ liệu ngẫu nhiên seed cố định.

## 2. Đề bài & ví dụ

Cho chuỗi `text` và chuỗi `pattern` dài **2**, đều gồm chữ thường. Bạn được chèn **đúng một** ký tự — hoặc `pattern[0]`, hoặc `pattern[1]` — vào **bất kỳ vị trí nào** của `text` (kể cả đầu và cuối).

Trả về số lần **lớn nhất** `pattern` xuất hiện như **dãy con** của chuỗi mới.

**Ví dụ 1:** `text = "abdcdbc"`, `pattern = "ac"` → `4`. Chèn `'a'` giữa `text[1]` và `text[2]`: `"abadcdbc"` có 4 dãy con `"ac"`. (`"aabdcdbc"` cũng cho 4.)

**Ví dụ 2:** `text = "aabb"`, `pattern = "ab"` → `6` (ví dụ `"aaabb"` hoặc `"aabbb"`).

**Ràng buộc:** `1 ≤ len(text) ≤ 10⁵`, `len(pattern) == 2`.

In [1]:
import random
import time
from itertools import combinations, product
from typing import List

examples = [("abdcdbc", "ac", 4), ("aabb", "ab", 6)]
for text, pat, exp in examples:
    print(f"text={text!r}, pattern={pat!r} -> kỳ vọng {exp}")

text='abdcdbc', pattern='ac' -> kỳ vọng 4
text='aabb', pattern='ab' -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1 Đếm dãy con "ab" bằng định nghĩa

Số dãy con `"ab"` = số cặp chỉ số `i < j` với `s[i] = 'a'`, `s[j] = 'b'`. Có thể đếm bằng `combinations` (chậm, `O(n²)`), dùng để kiểm tra.

In [2]:
def count_pairs_slow(s, a, b):
    return sum(1 for i, j in combinations(range(len(s)), 2) if s[i] == a and s[j] == b)

print(count_pairs_slow("abadcdbc", "a", "c"), count_pairs_slow("aabb", "a", "b"))

4 4


### 3.2 Đếm trong `O(n)`: mỗi `b` ghép với mọi `a` đứng trước

Duyệt trái sang phải, giữ `ca` = số `a` đã gặp. Gặp `b` ⇒ `ans += ca`. Gặp `a` ⇒ `ca += 1`. **Thứ tự** hai bước quan trọng khi `a == b` (xem 3.4).

In [3]:
def count_pairs(s, a, b):
    ca = ans = 0
    for ch in s:
        if ch == b:
            ans += ca          # ghép với mọi a đứng trước
        if ch == a:
            ca += 1
    return ans

print(count_pairs("abadcdbc", "a", "c"), count_pairs("aabb", "a", "b"))

4 4


### 3.3 Chèn ký tự vào chuỗi

`s[:p] + ch + s[p:]` chèn `ch` vào vị trí `p` (`0 ≤ p ≤ len(s)`): `p = 0` là đầu, `p = len(s)` là cuối. Có `len(s) + 1` vị trí.

In [4]:
s = "abc"
print([s[:p] + "X" + s[p:] for p in range(len(s) + 1)])

['Xabc', 'aXbc', 'abXc', 'abcX']


### 3.4 Khi `a == b`: đếm cặp trong cùng một loại

Nếu `pattern = "aa"`, số dãy con là `C(c, 2) = c(c−1)/2` với `c` = số chữ `a`. Trong vòng lặp ở 3.2, vì kiểm `b` **trước** rồi mới tăng `ca`, mỗi chữ `a` chỉ ghép với các chữ `a` **đứng trước** nó (không ghép với chính nó) ⇒ đúng `C(c, 2)`.

In [5]:
print(count_pairs("aaaa", "a", "a"), "= C(4,2) =", 4 * 3 // 2, "| kiểm chậm:", count_pairs_slow("aaaa", "a", "a"))

6 = C(4,2) = 6 | kiểm chậm: 6


## 4. Ý tưởng nền tảng / trực giác

Gọi `A = pattern[0]`, `B = pattern[1]`, `base` = số dãy con `AB` trong `text`. Chèn **một** ký tự chỉ tạo thêm các cặp **có chứa ký tự mới**; các cặp cũ giữ nguyên.

- Chèn `A` tại vị trí `p`: ký tự mới làm vế trái ⇒ thêm đúng **số `B` ở bên phải `p`**. Lớn nhất khi `p = 0` (đầu chuỗi): thêm `cntB` = tổng số `B`.
- Chèn `B` tại vị trí `p`: thêm **số `A` ở bên trái `p`**. Lớn nhất khi `p = n` (cuối chuỗi): thêm `cntA`.

⇒ Đáp án `= base + max(cntA, cntB)`.

Tính tay ví dụ 1 `text = "abdcdbc"`, `A = 'a'`, `B = 'c'`: `cntA = 1`, `cntC = 2`, `base = 2` (`a` ghép với hai chữ `c`). Đáp án `2 + max(1, 2) = 4` ✓ — chèn `'a'` ở đầu.

Ví dụ 2 `"aabb"`: `base = 4`, `cntA = cntB = 2` ⇒ `6` ✓.

**Vì sao chèn ở giữa không bao giờ tốt hơn?** Hình dung chữ `A` mới như một "người tìm bạn": nó chỉ kết bạn được với các chữ `B` **đứng sau** nó. Đặt nó càng sớm thì càng nhiều `B` đứng sau ⇒ đặt ở đầu là tốt nhất. Ngược lại chữ `B` mới chỉ kết bạn với các `A` **đứng trước** ⇒ đặt ở cuối. Bảng dưới liệt kê phần tăng thêm tại mọi vị trí của ví dụ 1 để thấy rõ hai dãy số đơn điệu.

In [6]:
text, pat = "abdcdbc", "ac"
A, B = pat
for p in range(len(text) + 1):
    left_A = text[:p].count(A); right_B = text[p:].count(B)
    print(f"p={p}: chèn '{A}' thêm {right_B} cặp | chèn '{B}' thêm {left_A} cặp")

p=0: chèn 'a' thêm 2 cặp | chèn 'c' thêm 0 cặp
p=1: chèn 'a' thêm 2 cặp | chèn 'c' thêm 1 cặp
p=2: chèn 'a' thêm 2 cặp | chèn 'c' thêm 1 cặp
p=3: chèn 'a' thêm 2 cặp | chèn 'c' thêm 1 cặp
p=4: chèn 'a' thêm 1 cặp | chèn 'c' thêm 1 cặp
p=5: chèn 'a' thêm 1 cặp | chèn 'c' thêm 1 cặp
p=6: chèn 'a' thêm 1 cặp | chèn 'c' thêm 1 cặp
p=7: chèn 'a' thêm 0 cặp | chèn 'c' thêm 1 cặp


## 5. Cách 1 — Brute Force / baseline: thử mọi vị trí chèn, đếm lại

**Ý tưởng:** với mỗi ký tự `ch ∈ {A, B}` và mỗi vị trí `p ∈ [0, n]`, dựng chuỗi mới, đếm dãy con bằng `count_pairs` (`O(n)`).

In [7]:
def max_subseq_brute(text, pattern):
    A, B = pattern
    best = 0
    for ch in (A, B):
        for p in range(len(text) + 1):
            best = max(best, count_pairs(text[:p] + ch + text[p:], A, B))
    return best

for text, pat, exp in examples:
    got = max_subseq_brute(text, pat); print(got, "kỳ vọng", exp); assert got == exp

4 kỳ vọng 4
6 kỳ vọng 6


**Độ phức tạp:** `2(n+1)` chuỗi, mỗi chuỗi dựng và đếm `O(n)` ⇒ `O(n²)` — với `n = 10⁵` là `2·10¹⁰` bước, TLE. Space `O(n)`.

**Nút thắt:** việc chèn chỉ thêm các cặp chứa ký tự mới; phần còn lại `base` không đổi — không cần đếm lại từ đầu.

In [8]:
rng = random.Random(0)
for n in (500, 1000, 2000):
    t_ = "".join(rng.choice("ab") for _ in range(n))
    t = time.perf_counter(); max_subseq_brute(t_, "ab")
    print(f"n={n}: {time.perf_counter() - t:.3f}s")

n=500: 0.017s
n=1000: 0.068s


n=2000: 0.283s


## 6. Ý tưởng thuật toán / bước đột phá

**Phân tích "phần tăng thêm":** sau khi chèn `ch` vào vị trí `p`, số dãy con mới `= base + gain(ch, p)` với
- `gain(A, p) = #B trong text[p:]` (ký tự mới làm vế trái), cộng thêm `[A == B] · #A trong text[:p]` nếu `A = B` (ký tự mới cũng có thể làm vế phải);
- `gain(B, p) = #A trong text[:p]` (+ phần tương ứng nếu `A = B`).

Khi `A ≠ B`: `gain(A, p)` **giảm** theo `p`, `gain(B, p)` **tăng** theo `p` ⇒ max tại hai đầu ⇒ `max(cntB, cntA)`.

Khi `A = B`: `gain = #A trái + #A phải = cntA` ở **mọi** vị trí ⇒ đáp án `C(c, 2) + c = C(c+1, 2)` — vẫn khớp công thức `base + max(cntA, cntB)` vì `cntA = cntB = c`.

**Quy tắc "nếu… thì…":**
- **Nếu** `cntA ≥ cntB` **thì** chèn `B` ở cuối; **ngược lại** chèn `A` ở đầu.
- Đếm `base` bằng một lượt, kiểm `B` **trước** khi tăng `cntA`.

**Invariant khi quét:** sau khi xử lý `text[0..i]`, `cntA` = số `A` trong đoạn đó, `base` = số cặp `AB` trong đoạn đó.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tiền tố/hậu tố: tính `gain` cho mọi vị trí trong `O(n)`

Không cần tin ngay vào tham lam: tính `left_A[p]` (số `A` trong `text[:p]`) và `right_B[p]` (số `B` trong `text[p:]`) rồi thử mọi `p` trong `O(1)` mỗi vị trí.

In [9]:
def max_subseq_prefix(text, pattern):
    A, B = pattern
    n = len(text)
    base = count_pairs(text, A, B)
    left_A = [0] * (n + 1)
    for i, ch in enumerate(text):
        left_A[i + 1] = left_A[i] + (ch == A)
    right_B = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        right_B[i] = right_B[i + 1] + (text[i] == B)
    best = 0
    for p in range(n + 1):
        gainA = right_B[p] + (left_A[p] if A == B else 0)     # chèn A tại p
        gainB = left_A[p] + (right_B[p] if A == B else 0)     # chèn B tại p
        best = max(best, gainA, gainB)
    return base + best

for text, pat, exp in examples:
    assert max_subseq_prefix(text, pat) == exp
print([max_subseq_prefix(t, p) for t, p, _ in examples], "| 'aaa','aa' ->", max_subseq_prefix("aaa", "aa"))

[4, 6] | 'aaa','aa' -> 6


**Độ phức tạp Cách 2:** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.2 Cách 3 — Tham lam: `base + max(cntA, cntB)`

In [10]:
def max_subseq_greedy(text, pattern):
    A, B = pattern
    base = count_pairs(text, A, B)
    return base + max(text.count(A), text.count(B))

for text, pat, exp in examples:
    assert max_subseq_greedy(text, pat) == exp
print([max_subseq_greedy(t, p) for t, p, _ in examples])

[4, 6]


**Độ phức tạp Cách 3:** ba lượt `O(n)` (đếm `base`, `count(A)`, `count(B)`), Space `O(1)`.

### 7.3 Cách 4 — Một lượt duy nhất (bản nộp chính)

Gộp tất cả vào một vòng: `cntA`, `cntB`, `base`. Cuối cùng `base + max(cntA, cntB)`.

In [11]:
def max_subseq_one_pass(text, pattern):
    A, B = pattern
    cntA = cntB = base = 0
    for ch in text:
        if ch == B:
            base += cntA
            cntB += 1
        if ch == A:
            cntA += 1
    return base + max(cntA, cntB)

for text, pat, exp in examples:
    assert max_subseq_one_pass(text, pat) == exp
print([max_subseq_one_pass(t, p) for t, p, _ in examples])

[4, 6]


In [12]:
class SolutionOnePass:
    def maximumSubsequenceCount(self, text: str, pattern: str) -> int:
        a, b = pattern
        ca = cb = base = 0
        for ch in text:
            if ch == b:
                base += ca
                cb += 1
            if ch == a:
                ca += 1
        return base + max(ca, cb)

print([SolutionOnePass().maximumSubsequenceCount(t, p) for t, p, _ in examples])

[4, 6]


**Độ phức tạp Cách 4:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.4 Cách 5 — "Chèn thật" rồi đếm: thử đúng hai ứng viên

Một cách kiểm chứng trực quan cho tham lam: chỉ thử **hai** chuỗi `A + text` và `text + B`, đếm bằng `count_pairs`, lấy max. `O(n)` và dễ thuyết phục người đọc nhất.

In [13]:
def max_subseq_two_candidates(text, pattern):
    A, B = pattern
    return max(count_pairs(A + text, A, B), count_pairs(text + B, A, B))

for text, pat, exp in examples:
    assert max_subseq_two_candidates(text, pat) == exp
print([max_subseq_two_candidates(t, p) for t, p, _ in examples])

[4, 6]


**Độ phức tạp Cách 5:** `O(n)` (hai lần đếm + nối chuỗi), Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 4 trên ví dụ 1 `text = "abdcdbc"`, `pattern = "ac"`:

| i | ch | là B ('c')? base += cntA | là A ('a')? cntA += 1 | cntA | cntB | base |
|---|---|---|---|---|---|---|
| 0 | a | | ✓ | 1 | 0 | 0 |
| 1 | b | | | 1 | 0 | 0 |
| 2 | d | | | 1 | 0 | 0 |
| 3 | c | ✓ +1 | | 1 | 1 | 1 |
| 4 | d | | | 1 | 1 | 1 |
| 5 | b | | | 1 | 1 | 1 |
| 6 | c | ✓ +1 | | 1 | 2 | 2 |

Đáp án `2 + max(1, 2) = 4` — chèn `'a'` ở đầu.

In [14]:
def trace_one_pass(text, pattern):
    A, B = pattern; cntA = cntB = base = 0
    for i, ch in enumerate(text):
        note = []
        if ch == B: base += cntA; cntB += 1; note.append(f"B: base += {cntA}")
        if ch == A: cntA += 1; note.append("A: cntA += 1")
        print(f"i={i} ch={ch!r}: {', '.join(note) or '-':24s} cntA={cntA} cntB={cntB} base={base}")
    choice = f"chèn '{B}' ở cuối" if cntA >= cntB else f"chèn '{A}' ở đầu"
    print(f"=> {base} + max({cntA}, {cntB}) = {base + max(cntA, cntB)} ({choice})")
    return base + max(cntA, cntB)

trace_one_pass("abdcdbc", "ac")
print()
trace_one_pass("aaa", "aa")

i=0 ch='a': A: cntA += 1             cntA=1 cntB=0 base=0
i=1 ch='b': -                        cntA=1 cntB=0 base=0
i=2 ch='d': -                        cntA=1 cntB=0 base=0
i=3 ch='c': B: base += 1             cntA=1 cntB=1 base=1
i=4 ch='d': -                        cntA=1 cntB=1 base=1
i=5 ch='b': -                        cntA=1 cntB=1 base=1
i=6 ch='c': B: base += 1             cntA=1 cntB=2 base=2
=> 2 + max(1, 2) = 4 (chèn 'a' ở đầu)

i=0 ch='a': B: base += 0, A: cntA += 1 cntA=1 cntB=1 base=0
i=1 ch='a': B: base += 1, A: cntA += 1 cntA=2 cntB=2 base=1
i=2 ch='a': B: base += 2, A: cntA += 1 cntA=3 cntB=3 base=3
=> 3 + max(3, 3) = 6 (chèn 'a' ở cuối)


6

In [15]:
def trace_gain_table(text, pattern):
    A, B = pattern; n = len(text)
    for p in range(n + 1):
        new_A = count_pairs(text[:p] + A + text[p:], A, B); new_B = count_pairs(text[:p] + B + text[p:], A, B)
        print(f"p={p}: chèn '{A}' -> {new_A:2d} | chèn '{B}' -> {new_B:2d}")

trace_gain_table("aabb", "ab")

p=0: chèn 'a' ->  6 | chèn 'b' ->  4
p=1: chèn 'a' ->  6 | chèn 'b' ->  5
p=2: chèn 'a' ->  6 | chèn 'b' ->  6
p=3: chèn 'a' ->  5 | chèn 'b' ->  6
p=4: chèn 'a' ->  4 | chèn 'b' ->  6


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def maximumSubsequenceCount(self, text: str, pattern: str) -> int:
        a, b = pattern
        ca = cb = base = 0
        for ch in text:
            if ch == b:            # kiểm vế phải TRƯỚC (đúng cả khi a == b)
                base += ca         # ghép với mọi a đứng trước
                cb += 1
            if ch == a:
                ca += 1
        # chèn a ở đầu (thêm cb cặp) hoặc chèn b ở cuối (thêm ca cặp)
        return base + max(ca, cb)

for text, pat, exp in examples:
    assert Solution().maximumSubsequenceCount(text, pat) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Phân rã số cặp.** Gọi `s'` là chuỗi sau khi chèn ký tự `x` ở vị trí `p`. Các cặp `(i < j)` của `s'` tạo thành dãy con `AB` chia làm hai loại: (1) cả hai chỉ số thuộc `text` — đúng `base` cặp, không phụ thuộc `p`; (2) một trong hai là ký tự mới. Loại (2):
- nếu `x = A` (và `A ≠ B`): ký tự mới chỉ có thể là vế trái ⇒ số cặp = số `B` sau vị trí `p` = `#B(text[p:]) ≤ cntB`, đạt bằng khi `p = 0`;
- nếu `x = B` (và `A ≠ B`): số cặp = `#A(text[:p]) ≤ cntA`, đạt bằng khi `p = n`;
- nếu `A = B`: ký tự mới ghép với mọi `A` (trước nó làm vế phải, sau nó làm vế trái) ⇒ đúng `cntA` cặp ở mọi `p`.

Vậy `max_{x, p} |s'|_{AB} = base + max(cntA, cntB)` trong mọi trường hợp.

**Đúng của vòng đếm.** Bất biến ở mục 6: khi gặp `text[j] = B`, `ca` = số `A` ở chỉ số `< j` (vì tăng `ca` **sau** khi xử lý vai trò `B`), nên `base` cộng đúng số cặp có vế phải là `j`. Tổng trên mọi `j` = `base`.

In [17]:
# Kiểm chứng gain đơn điệu (A != B) và hằng số (A == B) trên dữ liệu ngẫu nhiên
rng = random.Random(10)
for _ in range(300):
    text = "".join(rng.choice("abc") for _ in range(rng.randint(1, 15)))
    for A, B in [("a", "b"), ("b", "a"), ("a", "a")]:
        base = count_pairs(text, A, B)
        gA = [count_pairs(text[:p] + A + text[p:], A, B) - base for p in range(len(text) + 1)]
        gB = [count_pairs(text[:p] + B + text[p:], A, B) - base for p in range(len(text) + 1)]
        if A != B:
            assert all(gA[i] >= gA[i + 1] for i in range(len(gA) - 1)) and all(gB[i] <= gB[i + 1] for i in range(len(gB) - 1))
        else:
            assert len(set(gA)) == 1 and gA[0] == text.count(A)
print("gain(A, p) không tăng, gain(B, p) không giảm (A != B); gain hằng số khi A == B")

gain(A, p) không tăng, gain(B, p) không giảm (A != B); gain hằng số khi A == B


## 11. Phân tích Time / Space

Gọi `n = len(text)`.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi vị trí | `O(n²)` | `O(n)` |
| 2. Tiền tố/hậu tố | `O(n)` | `O(n)` |
| 3. Tham lam 3 lượt | `O(n)` | `O(1)` |
| 4. Một lượt | `O(n)` | `O(1)` |
| 5. Hai ứng viên | `O(n)` | `O(n)` |

Đáp án có thể tới `≈ n²/4 + n` ≈ `2,5·10⁹` — vượt `int32` (Java/C++ cần `long`).

In [18]:
rng = random.Random(2207)
big = "".join(rng.choice("ab") for _ in range(10**5))
for name, fn in [("prefix", max_subseq_prefix), ("greedy", max_subseq_greedy), ("one_pass", max_subseq_one_pass),
                 ("two_cand", max_subseq_two_candidates), ("Solution", lambda t, p: Solution().maximumSubsequenceCount(t, p))]:
    t = time.perf_counter(); r = fn(big, "ab"); r2 = fn("a" * 10**5, "aa")
    print(f"{name:9s}: {r}, 'a'*1e5 với 'aa': {r2} ({time.perf_counter() - t:.3f}s)")

prefix   : 1245711534, 'a'*1e5 với 'aa': 5000050000 (0.045s)
greedy   : 1245711534, 'a'*1e5 với 'aa': 5000050000 (0.008s)
one_pass : 1245711534, 'a'*1e5 với 'aa': 5000050000 (0.009s)
two_cand : 1245711534, 'a'*1e5 với 'aa': 5000050000 (0.014s)
Solution : 1245711534, 'a'*1e5 với 'aa': 5000050000 (0.009s)


## 12. Bộ test

- Ví dụ đề.
- Biên: `pattern` gồm hai chữ giống nhau; `text` không chứa chữ nào của `pattern` (đáp án 0); `text` chỉ chứa `A`; chỉ chứa `B`; `pattern = "ba"` ngược thứ tự; `n = 1`.
- Đối chiếu brute force trên dữ liệu ngẫu nhiên (bảng chữ nhỏ, cả `A == B`; seed cố định), và **vét cạn** mọi chuỗi độ dài ≤ 7 trên `{a, b, c}` với mọi pattern trên `{a, b}`.

In [19]:
fns = {"brute": max_subseq_brute, "prefix": max_subseq_prefix, "greedy": max_subseq_greedy,
       "one_pass": max_subseq_one_pass, "two_cand": max_subseq_two_candidates,
       "Solution": lambda t, p: Solution().maximumSubsequenceCount(t, p),
       "SolutionOnePass": lambda t, p: SolutionOnePass().maximumSubsequenceCount(t, p)}
edge = [("aaa", "aa", 6), ("xyz", "ab", 0), ("aaa", "ab", 3), ("bbb", "ab", 3), ("ab", "ba", 1),
        ("a", "aa", 1), ("b", "ab", 1), ("ba", "ab", 1)]
for text, pat, exp in [(t, p, e) for t, p, e in examples] + edge:
    for name, fn in fns.items():
        assert fn(text, pat) == exp, (name, text, pat, fn(text, pat))
cnt = 0
for L in range(1, 8):
    for tup in product("abc", repeat=L):
        text = "".join(tup)
        for pat in ("ab", "ba", "aa", "ac"):
            want = max_subseq_brute(text, pat)
            assert max_subseq_one_pass(text, pat) == want == max_subseq_prefix(text, pat)
            cnt += 1
print(f"Ví dụ + biên + vét cạn {cnt} bộ: OK")

Ví dụ + biên + vét cạn 13116 bộ: OK


In [20]:
rng = random.Random(42)
for _ in range(300):
    text = "".join(rng.choice("abcz") for _ in range(rng.randint(1, 40)))
    pat = rng.choice("abz") + rng.choice("abz")
    want = max_subseq_brute(text, pat)
    for name, fn in fns.items():
        assert fn(text, pat) == want, (name, text, pat)
print("300 test ngẫu nhiên: mọi cách khớp brute force")

300 test ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tăng `cntA` **trước** khi cộng (sai khi `A == B`)

Với `pattern = "aa"`, mỗi chữ `a` bị ghép với **chính nó**.

In [21]:
def wrong_order(text, pattern):
    a, b = pattern; ca = cb = base = 0
    for ch in text:
        if ch == a: ca += 1                    # SAI: tăng trước
        if ch == b: base += ca; cb += 1
    return base + max(ca, cb)

print("sai:", wrong_order("aaa", "aa"), "| đúng:", max_subseq_brute("aaa", "aa"))
print("với A != B vẫn đúng:", wrong_order("abdcdbc", "ac"))

sai: 9 | đúng: 6
với A != B vẫn đúng: 4


In [22]:
print("sửa:", Solution().maximumSubsequenceCount("aaa", "aa"))

sửa: 6


### Lỗi 2 — Luôn chèn `pattern[0]` ở đầu (quên so sánh)

Nếu số `A` nhiều hơn số `B`, chèn `B` ở cuối tốt hơn.

In [23]:
def wrong_always_A(text, pattern):
    a, b = pattern
    return count_pairs(a + text, a, b)          # SAI: chỉ một lựa chọn

print("sai:", wrong_always_A("aaab", "ab"), "| đúng:", max_subseq_brute("aaab", "ab"))

sai: 4 | đúng: 6


In [24]:
print("sửa:", Solution().maximumSubsequenceCount("aaab", "ab"))

sửa: 6


### Lỗi 3 — Đếm **xâu con liên tiếp** (`text.count(pattern)`) thay vì dãy con

In [25]:
def wrong_substring(text, pattern):
    return max((pattern[0] + text).count(pattern), (text + pattern[1]).count(pattern))   # SAI

print("sai:", wrong_substring("abdcdbc", "ac"), "| đúng:", 4)

sai: 0 | đúng: 4


In [26]:
print("sửa:", max_subseq_two_candidates("abdcdbc", "ac"))

sửa: 4


### Lỗi 4 — Chèn ở **giữa** "chỗ có nhiều nhất" (tham lam sai vị trí)

Ý nghĩ "chèn `A` ngay trước chữ `B` đầu tiên" cho cùng số cặp với chèn ở đầu chỉ khi không có `B` nào trước đó — nhưng "chèn `A` ngay trước chữ `B` **cuối cùng**" thì sai. Bản sai dưới đây chèn `A` ngay trước `B` cuối cùng.

In [27]:
def wrong_middle(text, pattern):
    a, b = pattern
    p = text.rfind(b)
    if p == -1:
        return count_pairs(text + b, a, b)
    return count_pairs(text[:p] + a + text[p:], a, b)            # SAI: chỉ thử chèn A trước B cuối cùng

print("sai:", wrong_middle("cbcbca", "cb"), "| đúng:", max_subseq_brute("cbcbca", "cb"))

sai: 4 | đúng: 6


In [28]:
print("sửa (chèn A ở ĐẦU):", Solution().maximumSubsequenceCount("cbcbca", "cb"))

sửa (chèn A ở ĐẦU): 6


### Lỗi 5 — Chọn ngược: "nhiều `A` thì chèn thêm `A`"

Trực giác "loại nào nhiều thì chèn thêm loại đó" là **sai**: chèn `A` ở đầu ghép với mọi **`B`** (thêm `cntB`), còn chèn `B` ở cuối ghép với mọi **`A`** (thêm `cntA`). Khi `A` nhiều hơn, phải chèn **`B`**. Lỗi này lọt qua ví dụ 2 vì ở đó `cntA = cntB`.

In [29]:
def wrong_swapped_choice(text, pattern):
    a, b = pattern; ca = cb = base = 0
    for ch in text:
        if ch == b: base += ca; cb += 1
        if ch == a: ca += 1
    return base + (cb if ca >= cb else ca)       # SAI: "nhiều A -> chèn A ở đầu" (chỉ thêm cb)

print("ví dụ 2 (tình cờ đúng):", wrong_swapped_choice("aabb", "ab"))
print("sai:", wrong_swapped_choice("aaab", "ab"), "| đúng:", max_subseq_brute("aaab", "ab"))

ví dụ 2 (tình cờ đúng): 6
sai: 4 | đúng: 6


In [30]:
print("sửa: nhiều A thì chèn B ở cuối ->", Solution().maximumSubsequenceCount("aaab", "ab"))

sửa: nhiều A thì chèn B ở cuối -> 6


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Đếm dãy con 2 ký tự** ⇒ một lượt: mỗi vế phải ghép với số vế trái đã gặp.
- **"Chèn/xóa một phần tử để tối đa"** ⇒ tách tổng thành phần **không đổi** + phần **phụ thuộc vị trí**, rồi tối ưu phần sau (thường đạt ở biên).
- Luôn thử trường hợp hai ký tự **giống nhau** — thứ tự cập nhật bộ đếm quyết định đúng/sai.
- Đáp án có thể vượt `int32`.
- Brute force `O(n²)` chèn từng vị trí rất dễ viết — dùng làm chuẩn.
- Dấu hiệu nhận dạng chung của nhóm bài "chèn/xóa một phần tử": **phần tăng/giảm** là hàm đơn điệu theo vị trí ⇒ tối ưu ở biên; nếu không đơn điệu, tính nó cho mọi vị trí bằng tiền tố/hậu tố (Cách 2) — vẫn `O(n)`.
- Khi đề hỏi "số lần xuất hiện như dãy con", hãy tự hỏi: mỗi lần xuất hiện được "đếm" tại ký tự **cuối** của nó — đó là gốc của mọi lời giải một lượt.

## 15. Biến thể của bài

1. **Đếm dãy con 3 ký tự** (`"abc"`): DP ba bộ đếm `ca, cab, cabc` (LeetCode 1987-kiểu, hoặc 2222 — Number of Ways to Select Buildings).
2. **LeetCode 115 — Distinct Subsequences (Hard):** pattern dài tùy ý — DP `O(n·m)`.
3. **Chèn `k` ký tự** (bài tập 5): chèn `x` chữ `A` ở đầu và `k − x` chữ `B` ở cuối; tối ưu theo `x`.
4. **Xóa một ký tự để tối thiểu số dãy con** — phần giảm phụ thuộc vị trí, xét từng ký tự.

Minh họa biến thể 4 (tự dựng, kiểm bằng brute force):

In [31]:
def min_after_delete(text, pattern):
    A, B = pattern; n = len(text)
    base = count_pairs(text, A, B)
    left_A = [0] * (n + 1); right_B = [0] * (n + 1)
    for i, ch in enumerate(text): left_A[i + 1] = left_A[i] + (ch == A)
    for i in range(n - 1, -1, -1): right_B[i] = right_B[i + 1] + (text[i] == B)
    best = base
    for i, ch in enumerate(text):
        loss = 0
        if ch == A: loss += right_B[i + 1]            # cặp mà ch làm vế trái
        if ch == B: loss += left_A[i]                 # cặp mà ch làm vế phải
        best = min(best, base - loss)
    return best

rng = random.Random(4)
for _ in range(400):
    text = "".join(rng.choice("abc") for _ in range(rng.randint(1, 12)))
    pat = rng.choice(["ab", "ba", "aa"])
    brute = min(count_pairs(text[:i] + text[i + 1:], *pat) for i in range(len(text)))
    assert min_after_delete(text, pat) == brute
print("Biến thể xóa một ký tự để tối thiểu khớp brute force; 'aabb','ab' ->", min_after_delete("aabb", "ab"))

Biến thể xóa một ký tự để tối thiểu khớp brute force; 'aabb','ab' -> 2


Minh họa biến thể 1 và 2 (tự dựng, kiểm bằng brute force):

- **Dãy con 3 ký tự `xyz`:** giữ ba bộ đếm `c1` = số `x`, `c2` = số dãy con `xy`, `c3` = số dãy con `xyz`. Gặp ký tự `ch`: **cập nhật từ bậc cao xuống thấp** (`z` rồi `y` rồi `x`) để không dùng ký tự hiện tại hai lần khi các chữ trùng nhau.
- **Pattern dài tùy ý (LeetCode 115):** `dp[j]` = số cách khớp `pattern[:j]`; duyệt `j` **giảm dần** với mỗi ký tự của `text`.

In [32]:
def count_subseq3(s, p):
    x, y, z = p; c1 = c2 = c3 = 0
    for ch in s:
        if ch == z: c3 += c2
        if ch == y: c2 += c1
        if ch == x: c1 += 1
    return c3

def count_subseq_any(s, p):
    dp = [1] + [0] * len(p)
    for ch in s:
        for j in range(len(p), 0, -1):
            if p[j - 1] == ch:
                dp[j] += dp[j - 1]
    return dp[-1]

rng = random.Random(115)
for _ in range(400):
    s = "".join(rng.choice("ab") for _ in range(rng.randint(1, 10)))
    p = "".join(rng.choice("ab") for _ in range(rng.randint(1, 3)))
    brute = sum(1 for idx in combinations(range(len(s)), len(p)) if "".join(s[i] for i in idx) == p)
    assert count_subseq_any(s, p) == brute
    if len(p) == 3: assert count_subseq3(s, p) == brute
print("Đếm dãy con 3 ký tự và pattern tùy ý khớp brute force; 'aaab' chứa 'aab':", count_subseq_any("aaab", "aab"), "lần")

Đếm dãy con 3 ký tự và pattern tùy ý khớp brute force; 'aaab' chứa 'aab': 3 lần


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi vị trí | chèn rồi đếm lại | `O(n²)` | `O(n)` | kiểm tra |
| 2. Tiền tố/hậu tố | `gain` mọi vị trí | `O(n)` | `O(n)` | không cần tin tham lam |
| 3. Tham lam 3 lượt | `base + max(cntA, cntB)` | `O(n)` | `O(1)` | dễ đọc |
| 4. Một lượt | gộp bộ đếm | `O(n)` | `O(1)` | **nộp bài** |
| 5. Hai ứng viên | `A+text`, `text+B` | `O(n)` | `O(n)` | giải thích trực quan |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `maximumSubsequenceCount("abab", "ab")`.
2. **Tính tay:** `text = "zzz"`, `pattern = "zz"` — đáp án và giải thích bằng `C(c+1, 2)`.
3. **Điền TODO:** `count_subseq2(s, a, b)` đếm dãy con `ab` trong `O(n)`.
4. **Tự viết lại:** viết lại Cách 4 không nhìn code.
5. **Mini challenge:** `max_after_k_insertions(text, pattern, k)` — chèn **đúng `k`** ký tự (mỗi ký tự là `A` hoặc `B`, vị trí tùy ý). Kiểm bằng brute force (chèn tham lam từng ký tự không đủ! — hãy thử mọi cách chia `x` chữ `A` ở đầu, `k − x` chữ `B` ở cuối, và so với vét cạn nhỏ).

In [33]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_subseq2(s, a, b):
    # TODO
    pass

In [34]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_max_subseq(text, pattern):
    # TODO
    pass

In [35]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def max_after_k_insertions(text, pattern, k):
    # TODO: thử x = 0..k: x chữ A ở đầu, k - x chữ B ở cuối
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) `base = 3` (cặp (0,1), (0,3), (2,3)). (2) `cntA = cntB = 2`. (3) `3 + 2 = 5`.

**Bài 2.** (1) `c = 3`, `base = C(3,2) = 3`. (2) Chèn một `z` ⇒ `c = 4` ⇒ `C(4,2) = 6`. (3) Công thức `3 + max(3, 3) = 6`.

**Bài 3.** (1) `ca = ans = 0`. (2) Gặp `b`: `ans += ca`. (3) Gặp `a`: `ca += 1` (sau bước 2).

**Bài 4.** (1) Ba bộ đếm. (2) Kiểm `b` trước `a`. (3) `base + max(ca, cb)`.

**Bài 5.** (1) Với `A ≠ B`: chèn `x` chữ `A` ở đầu và `k − x` chữ `B` ở cuối tạo thêm `x·cntB + (k−x)·cntA + x(k−x)` cặp. (2) Thử mọi `x`. (3) Với `A = B`: tổng chữ là `c + k` ⇒ `C(c + k, 2)`.

## 19. Đáp án tham khảo

In [36]:
# Bài 1, 2
assert Solution().maximumSubsequenceCount("abab", "ab") == 5
assert Solution().maximumSubsequenceCount("zzz", "zz") == 6
print("Bài 1: 5 | Bài 2: 6")

Bài 1: 5 | Bài 2: 6


In [37]:
# Bài 3, 4
count_subseq2_ans = count_pairs
my_max_subseq_ans = max_subseq_one_pass
rng = random.Random(3)
for _ in range(300):
    s = "".join(rng.choice("ab") for _ in range(rng.randint(1, 20)))
    a, b = rng.choice("ab"), rng.choice("ab")
    assert count_subseq2_ans(s, a, b) == count_pairs_slow(s, a, b)
    assert my_max_subseq_ans(s, a + b) == max_subseq_brute(s, a + b)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [38]:
# Bài 5
def max_after_k_insertions_ans(text, pattern, k):
    A, B = pattern
    if A == B:
        c = text.count(A) + k
        return c * (c - 1) // 2
    base = count_pairs(text, A, B); cA, cB = text.count(A), text.count(B)
    return base + max(x * cB + (k - x) * cA + x * (k - x) for x in range(k + 1))

def k_insert_brute(text, pattern, k):
    A, B = pattern
    frontier = {text}
    for _ in range(k):
        frontier = {s[:p] + ch + s[p:] for s in frontier for ch in {A, B} for p in range(len(s) + 1)}
    return max(count_pairs(s, A, B) for s in frontier)

rng = random.Random(5)
for _ in range(150):
    text = "".join(rng.choice("abc") for _ in range(rng.randint(1, 6)))
    pat = rng.choice(["ab", "ba", "aa"]); k = rng.randint(1, 3)
    assert max_after_k_insertions_ans(text, pat, k) == k_insert_brute(text, pat, k)
print("Bài 5: OK; 'abdcdbc','ac', k=3 ->", max_after_k_insertions_ans("abdcdbc", "ac", 3))

Bài 5: OK; 'abdcdbc','ac', k=3 -> 9


**Giải thích thêm cho Bài 5.** Với `A ≠ B`, chèn `x` chữ `A` (đều ở đầu) và `k − x` chữ `B` (đều ở cuối) tạo thêm: `x·cntB` (mỗi `A` mới ghép mọi `B` cũ), `(k − x)·cntA` (mỗi `B` mới ghép mọi `A` cũ), và `x·(k − x)` (mỗi `A` mới ghép mỗi `B` mới, vì `A` mới đứng trước `B` mới). Hàm theo `x` là parabol úp ⇒ chỉ cần thử `x = 0..k`. Bảng dưới in mọi `x` cho ví dụ 1 với `k = 3`.

In [39]:
text, (A, B), k = "abdcdbc", "ac", 3
base, cA, cB = count_pairs(text, A, B), text.count(A), text.count(B)
for x in range(k + 1):
    gain = x * cB + (k - x) * cA + x * (k - x)
    print(f"x={x} chữ '{A}' ở đầu, {k - x} chữ '{B}' ở cuối: thêm {x}*{cB} + {k - x}*{cA} + {x}*{k - x} = {gain} -> tổng {base + gain}")

x=0 chữ 'a' ở đầu, 3 chữ 'c' ở cuối: thêm 0*2 + 3*1 + 0*3 = 3 -> tổng 5
x=1 chữ 'a' ở đầu, 2 chữ 'c' ở cuối: thêm 1*2 + 2*1 + 1*2 = 6 -> tổng 8
x=2 chữ 'a' ở đầu, 1 chữ 'c' ở cuối: thêm 2*2 + 1*1 + 2*1 = 7 -> tổng 9
x=3 chữ 'a' ở đầu, 0 chữ 'c' ở cuối: thêm 3*2 + 0*1 + 3*0 = 6 -> tổng 8


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
a, b = pattern
ca = cb = base = 0
for ch in text:
    if ch == b: base += ca; cb += 1     # vế phải TRƯỚC
    if ch == a: ca += 1
return base + max(ca, cb)              # a ở đầu (thêm cb) hoặc b ở cuối (thêm ca)
```

In [40]:
def template(text, pattern):
    a, b = pattern; ca = cb = base = 0
    for ch in text:
        if ch == b: base += ca; cb += 1
        if ch == a: ca += 1
    return base + max(ca, cb)
print([template(t, p) for t, p, _ in examples])

[4, 6]


**Checklist:**

- [ ] Đếm **dãy con** (không phải xâu con liên tiếp).
- [ ] Kiểm vế phải `b` **trước** khi tăng `ca` (đúng khi `a == b`).
- [ ] So sánh hai lựa chọn: chèn `a` ở đầu (`+cb`) và chèn `b` ở cuối (`+ca`).
- [ ] Đáp án có thể vượt `int32`.
- [ ] Đã thử: `pattern` hai chữ giống nhau, `text` không chứa chữ nào của `pattern`.